# 🚀 04: Retraining Experiments Suite

> **Self-Contained Repository Notebook** (`drone-telemetry-unsupervised/notebooks/04_retraining_experiments.ipynb`)
>
> **Research Focus**: Trains and validates all **Retraining Models**, displays **Per-Seed** and **Per-Fold** breakdown tables, saves all trained model checkpoints (`.pth`), and automatically compresses everything into a downloadable ZIP archive.

---

### 📦 Experiments Executed in this Notebook:
1. **EXP-8: Multi-Seed Stability Test (5 Seeds: 17, 42, 137, 521, 2026)**
   - Retrains TCN on Ultimate 9 across 5 seeds.
   - Saves model weights (`tcn_seed_*.pth`).
   - Displays **Per-Seed Breakdown Table** + Mean ± Std / Min / Max.
2. **EXP-9: Feature Bloat Validation (Paired Multi-Seed & Hypothesis Testing)**
   - Retrains TCN on Ultimate 9 vs Ultimate 9 + Yaw across the 5 seeds.
   - Saves model weights (`tcn_yaw_seed_*.pth`).
   - Displays **Per-Seed Paired Comparison Table** + Paired t-test and Wilcoxon statistical tests.
3. **EXP-10: 5-Fold Flight-Wise Cross-Validation**
   - Stratified 5-Fold Cross-Validation by Flight ID on genuine DJI flights.
   - Saves fold model weights (`tcn_fold_*.pth`).
   - Displays **Per-Fold Breakdown Table** (Folds 1 to 5) + CV Mean ± Std.
4. **EXP-13: Device-Wise Split Benchmark (Cross-Hardware Zero-Shot Generalization)**
   - Partitions genuine DJI flights by physical drone model (trains on 7 models, tests on 2 unseen models).
   - Evaluates all 12 models (8 Pointwise Classical + 4 Deep Sequence Autoencoders) on Ultimate 9 and Baseline 7.
   - Displays **Per-Class Cross-Hardware Summary Table**.
5. **EXP-12: Additional & Optional Feature Set Ablations (Exploratory)**
   - Retrains TCN on isolated feature sets (`pos_only`, `pe_only`, `entropy_only`, `pe_autocorr`).
   - Displays comparative ablation results table.
6. **Automatic Packaging**:
   - Compresses all newly trained `.pth` model weights, CSV tables, and 300 DPI plots into `retraining_experiments.zip`.

---
### 📁 Output Location:
All model checkpoints, CSV tables, plots, and the ZIP bundle are automatically saved to:
`implement/output/retraining_experiments/`



## 🛠️ Step 0: Environment Setup & Repository Anchor


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch

# Idempotent repository root anchor
current = Path.cwd().resolve()
while current != current.parent and not (current / "implement").exists():
    current = current.parent
PROJECT_ROOT = current
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from implement.utils.helper import get_output_dir
out_dir = get_output_dir() / "retraining_experiments"
out_dir.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Repository Root: {PROJECT_ROOT}")
print(f"✅ Output Save Directory: {out_dir}")
print(f"✅ Compute Device: {device} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)}")



---
## 🎲 EXP-8: Multi-Seed Stability Test (5 Seeds: 17, 42, 137, 521, 2026)

### Purpose
Tests whether the headline detection performance on **Ultimate 9** is statistically stable across different random data splits and model weight initializations.

### Execution
Trains TCN Autoencoder on Seeds `[17, 42, 137, 521, 2026]`, saves all 5 model weights to `models/tcn_seed_{seed}.pth`, and produces a complete **Per-Seed Breakdown Table**.



In [ ]:
from presets.run_retraining_suite import run_exp8_multi_seed

# Run Multi-Seed Stability Test (5 Seeds)
df_exp8_seeds, df_exp8_full = run_exp8_multi_seed(
    seeds=[17, 42, 137, 521, 2026],
    epochs=15,
    patience=7,
    k_thresh=3.0
)

print("\n📊 EXP-8 COMPLETE MULTI-SEED RESULTS TABLE (Per-Seed Breakdown & Summary):")
display(df_exp8_full)



---
## 🔬 EXP-9: Feature Bloat Validation (Paired Multi-Seed & Hypothesis Testing)

### Purpose
Tests whether adding `yaw_acceleration` degrades anomaly detection performance due to double-derivative numerical noise amplification.

### Execution
Trains TCN on **Ultimate 9** vs **Ultimate 9 + Yaw** across the exact same 5 seeds, saves all model checkpoints (`models/tcn_yaw_seed_{seed}.pth`), and displays the **Per-Seed Paired Comparison Table** with statistical hypothesis tests (Paired t-test and Wilcoxon signed-rank test).



In [ ]:
from presets.run_retraining_suite import run_exp9_feature_bloat

# Run Paired Multi-Seed Feature Bloat Validation
df_exp9_paired = run_exp9_feature_bloat(
    seeds=[17, 42, 137, 521, 2026],
    epochs=15,
    patience=7,
    k_thresh=3.0
)

print("\n📊 EXP-9 PAIRED FEATURE BLOAT TABLE (Per-Seed Delta & Statistical Tests):")
display(df_exp9_paired)



---
## 🔄 EXP-10: 5-Fold Flight-Wise Cross-Validation

### Purpose
Provides a rigorous cross-validation estimate of detection and false alarm generalization across physical flight diversity, stratified by `flight_id`.

### Execution
Performs 5-Fold Cross-Validation on genuine DJI flights, trains a fresh TCN model for each fold, saves checkpoints (`models/tcn_fold_{fold_idx}.pth`), and outputs the **Per-Fold Breakdown Table** (Folds 1 to 5) alongside the Cross-Validation Mean ± Std.



In [ ]:
from presets.run_retraining_suite import run_exp10_flight_cv

# Run 5-Fold Flight-Wise Cross-Validation
df_exp10_folds, df_exp10_full = run_exp10_flight_cv(
    n_splits=5,
    epochs=15,
    patience=7,
    k_thresh=3.0
)

print("\n📊 EXP-10 5-FOLD CROSS-VALIDATION RESULTS TABLE (Per-Fold Breakdown & Mean ± Std):")
display(df_exp10_full)



---
## 🚁 EXP-13: Device-Wise Split Benchmark (Cross-Hardware Zero-Shot Generalization)

### Purpose
Evaluates zero-shot cross-hardware generalization. The detector is trained exclusively on 7 DJI drone models (`inspire_1`, `phantom_3`, `matrice_210`, `phantom_4_pro_v2`, `mavic_2`, `matrice_600`, `mavic_pro`), validated on 1 model (`mavic_air`), and tested on 2 completely unseen drone airframes (`phantom_4`, `inspire_2`) + all spoofed attack classes.

### Execution
Evaluates all 12 unsupervised models (8 Classical Pointwise + 4 Deep Sequence Autoencoders) on Ultimate 9 and Baseline 7.



In [ ]:
from presets.run_retraining_suite import run_exp13_device_split

# Run Device-Wise Hardware Split Benchmark
df_exp13_u9, df_exp13_b7 = run_exp13_device_split()

print("\n📊 EXP-13 Ultimate 9 Device-Wise Split Results:")
display(df_exp13_u9)

print("\n📊 EXP-13 Baseline 7 Device-Wise Split Results:")
display(df_exp13_b7)



---
## 🧪 EXP-12: Additional & Optional Feature Set Ablations (Exploratory)

### Purpose
Fills all remaining gaps in the ablation story by isolating the individual contributions of:
1. `baseline_7_plus_pos_only` (`position_residual_std` without `prediction_error`)
2. `pe_only` (`prediction_error` alone)
3. `baseline_7_plus_entropy_only` (`speed_spectral_entropy` without residuals)
4. `baseline_7_plus_pe_autocorr` (`prediction_error` + autocorrelation)



In [ ]:
from presets.run_retraining_suite import run_exp12_additional_feature_ablations

# Run Optional & Exploratory Feature Set Ablations
df_extra_ablations = run_exp12_additional_feature_ablations()

print("\n📊 EXP-12 Additional Feature Ablations Summary:")
display(df_extra_ablations)



---
## 📦 Step 6: Bundle & Export All Checkpoints, Tables and Plots

Compresses all newly trained `.pth` model weights, scalers, results CSVs, and 300 DPI figures into a single downloadable ZIP archive.



In [ ]:
from presets.run_retraining_suite import zip_tier2_results

# Create downloadable ZIP archive of all Retraining outputs
zip_path = zip_tier2_results()
print(f"🎉 Retraining Experiments Archive Ready for Download: {zip_path}")

